In [1]:
# Cell 1 — Load both parquets for a single state
from pathlib import Path
import pandas as pd

speeds_dir = Path("/home/eprashar_solutions_corelogic_com/network-idx/data/processed/fcc/speeds")

block_df = pd.read_parquet(speeds_dir / "fcc_fixed_speeds_AK_02.parquet")
provider_df = pd.read_parquet(speeds_dir / "fcc_fixed_speeds_providers_AK_02.parquet")

# Derive tract_geoid (first 11 digits of block_geoid)
block_df["tract_geoid"] = block_df["block_geoid"].str[:11]
provider_df["tract_geoid"] = provider_df["block_geoid"].str[:11]

print(f"Block rows: {len(block_df):,}, Provider rows: {len(provider_df):,}")
print(f"Unique tracts: {block_df['tract_geoid'].nunique()}")

Block rows: 10,499, Provider rows: 17,218
Unique tracts: 175


In [2]:
# Cell 2 — Aggregate block parquet to tract: SUM counts, MAX speeds
TECHS = ["cable", "copper", "fiber"]

agg_dict = {
    "state_usps": ("state_usps", "first"),
    "state_fips": ("state_fips", "first"),
}
for tech in TECHS:
    agg_dict[f"{tech}_location_count"] = (f"{tech}_location_count", "sum")
    agg_dict[f"{tech}_max_download_speed"] = (f"{tech}_max_download_speed", "max")
    agg_dict[f"{tech}_max_upload_speed"] = (f"{tech}_max_upload_speed", "max")

tract_speeds = block_df.groupby("tract_geoid", as_index=False).agg(**agg_dict)
print(f"Tract-level speed rows: {len(tract_speeds)}")
tract_speeds.head()

Tract-level speed rows: 175


,tract_geoid,state_usps,state_fips,cable_location_count,cable_max_download_speed,cable_max_upload_speed,copper_location_count,copper_max_download_speed,copper_max_upload_speed,fiber_location_count,fiber_max_download_speed,fiber_max_upload_speed
0,02013000100,AK,02,0.0,NaN,NaN,753.0,0.0,0.0,837.0,2500.0,75.0
1,02016000100,AK,02,0.0,NaN,NaN,0.0,NaN,NaN,203.0,10.0,1.0
2,02016000200,AK,02,568.0,0.0,0.0,0.0,NaN,NaN,618.0,2500.0,75.0
3,02020000101,AK,02,1824.0,2500.0,75.0,2013.0,120.0,60.0,228.0,1000.0,500.0
4,02020000102,AK,02,1561.0,2500.0,75.0,1790.0,120.0,60.0,388.0,1000.0,500.0


In [5]:
# Cell 3 — Aggregate provider parquet to tract: NUNIQUE provider_id per tech
provider_counts = pd.DataFrame({"tract_geoid": provider_df["tract_geoid"].unique()})
for tech in TECHS:
    loc_col = f"{tech}_location_count"
    # Filter to rows where this provider actually serves this tech in a block
    tech_providers = (
        provider_df[provider_df[loc_col].notna()]
        .groupby("tract_geoid")["provider_id"]
        .nunique()
        .rename(f"{tech}_provider_count")
    )
    provider_counts = provider_counts.merge(tech_providers, on="tract_geoid", how="left")

print(f"Provider count rows: {len(provider_counts)}")
provider_counts.sort_values(by="fiber_provider_count",ascending=False).head()


Provider count rows: 175


,tract_geoid,cable_provider_count,copper_provider_count,fiber_provider_count
156,02185000100,1.0,NaN,3.0
74,02090000100,1.0,1.0,3.0
93,02090001902,1.0,1.0,3.0
82,02090000900,1.0,1.0,3.0
83,02090001000,1.0,1.0,3.0


In [6]:
# Cell 4 — Merge speeds + provider counts into final tract-level DataFrame
tract_df = tract_speeds.merge(provider_counts, on="tract_geoid", how="left")

# Reorder to match desired output schema
output_cols = [
    "tract_geoid",
    "state_usps",
    "state_fips",
    "cable_location_count",
    "cable_provider_count",
    "cable_max_download_speed",
    "cable_max_upload_speed",
    "copper_location_count",
    "copper_provider_count",
    "copper_max_download_speed",
    "copper_max_upload_speed",
    "fiber_location_count",
    "fiber_provider_count",
    "fiber_max_download_speed",
    "fiber_max_upload_speed",
]

tract_df = tract_df[output_cols]
print(f"Final tract DataFrame: {tract_df.shape}")
tract_df.head(10)

Final tract DataFrame: (175, 15)


,tract_geoid,state_usps,state_fips,cable_location_count,cable_provider_count,cable_max_download_speed,cable_max_upload_speed,copper_location_count,copper_provider_count,copper_max_download_speed,copper_max_upload_speed,fiber_location_count,fiber_provider_count,fiber_max_download_speed,fiber_max_upload_speed
0,02013000100,AK,02,0.0,NaN,NaN,NaN,753.0,1.0,0.0,0.0,837.0,1.0,2500.0,75.0
1,02016000100,AK,02,0.0,NaN,NaN,NaN,0.0,NaN,NaN,NaN,203.0,1.0,10.0,1.0
2,02016000200,AK,02,568.0,1.0,0.0,0.0,0.0,NaN,NaN,NaN,618.0,1.0,2500.0,75.0
3,02020000101,AK,02,1824.0,1.0,2500.0,75.0,2013.0,1.0,120.0,60.0,228.0,1.0,1000.0,500.0
4,02020000102,AK,02,1561.0,1.0,2500.0,75.0,1790.0,1.0,120.0,60.0,388.0,1.0,1000.0,500.0
5,02020000201,AK,02,1292.0,1.0,2500.0,75.0,1276.0,1.0,120.0,60.0,319.0,1.0,1000.0,500.0
6,02020000202,AK,02,1985.0,1.0,2500.0,75.0,2091.0,1.0,120.0,60.0,175.0,2.0,1000.0,500.0
7,02020000204,AK,02,563.0,1.0,2500.0,75.0,1264.0,2.0,120.0,60.0,304.0,1.0,1000.0,500.0
8,02020000205,AK,02,2472.0,1.0,2500.0,75.0,2352.0,1.0,120.0,60.0,349.0,1.0,1000.0,500.0
9,02020000206,AK,02,1089.0,1.0,2500.0,75.0,1197.0,1.0,120.0,60.0,65.0,1.0,1000.0,500.0


In [10]:
# Cell 5 — Sanity checks
print("=== Row counts ===")
print(f"Blocks: {len(block_df):,}  →  Tracts: {len(tract_df):,}")

print("\n=== Cable location count: block sum vs tract sum ===")
print(f"Block total:  {block_df['cable_location_count'].sum():,.0f}")
print(f"Tract total:  {tract_df['cable_location_count'].sum():,.0f}")


for element in ["provider_count", "location_count", "max_download_speed","max_upload_speed"]:
    print(f"\n=== Sample {element} description ===")
    print(tract_df[[f"cable_{element}", f"copper_{element}", f"fiber_{element}"]].describe())

=== Row counts ===
Blocks: 10,499  →  Tracts: 175

=== Cable location count: block sum vs tract sum ===
Block total:  163,821
Tract total:  163,821

=== Sample provider_count description ===
       cable_provider_count  copper_provider_count  fiber_provider_count
count                 134.0             171.000000            158.000000
mean                    1.0               1.093567              1.253165
std                     0.0               0.311570              0.528629
min                     1.0               1.000000              1.000000
25%                     1.0               1.000000              1.000000
50%                     1.0               1.000000              1.000000
75%                     1.0               1.000000              1.000000
max                     1.0               3.000000              3.000000

=== Sample location_count description ===
       cable_location_count  copper_location_count  fiber_location_count
count            175.000000         